In [1]:
import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import visualization.visualization as vis
from nemi import NEMI, SingleNemi

import torch
import numpy as np

import colorsys
from einops import rearrange
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

import itertools

/home/jovyan/conda_envs/main_cuml/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/home/jovyan/conda_envs/main_cuml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
source = cutouts_dataset.CutoutDataSource(bucket="dbof", folder="cutouts_dataset_v2",
                                          run_id="1_00",
                                          dataset_name="cutout_dataset.zarr",
                                          s3_endpoint="https://s3-west.nrp-nautilus.io"
                                          )

source.print_available_channels()

23 available channels:
['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f','oceQnet','ekman_pumping','wind_stress_curl','SIarea','XC','YC']


In [3]:
#data_channels = ['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f','oceQnet','ekman_pumping','wind_stress_curl']

data_channels_small = ['Eta','Salt','Theta','U','V','gradb2','oceQnet','wind_stress_curl']
#data_channels_engineered = ['gradb2','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']
dataset = cutouts_dataset.CutoutDataset.from_source(data_channels=data_channels_small, source=source, subset=False,
                                                    subsample_per_chunk=64, num_sample_chunks=1, n_workers=4)

/home/jovyan/conda_envs/main_cuml/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44853 instead
  warnings.warn(


<Client: 'tcp://127.0.0.1:34795' processes=4 threads=64, memory=256.00 GiB>
nrp link url : https://jupyterhub-west.nrp-nautilus.io/hub/user-redirect/proxy/44853/status
dropped 0 ice, 0 NaN; kept 3250 / 3250
features ['Eta', 'Salt', 'Theta', 'U', 'V', 'gradb2', 'oceQnet', 'wind_stress_curl'] | coords ['XC', 'YC']


2026-08-03 13:37:33,701 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-08-03 13:37:33,703 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-08-03 13:37:33,705 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-08-03 13:37:33,705 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing


In [4]:
import math

patch_size = 8
patches = dataset.get_patches(patch_size=patch_size)   # (N_patches, C_feat*p*p), coords excluded

print(patches.shape)
print(math.log(patches.shape[0]) + 15)

(208000, 512)
27.245293358683455


In [7]:
EMBEDDING_DIMENSIONS = 3
ENSEMBLE_MEMEBERS = 8

In [ ]:
## Run NEMI on the patches (GPU: cuML UMAP + clustering)
nemi = NEMI(params={
    "device": "gpu",
    "embedding_dict":  {"n_components": 3, "n_neighbors": 57, "min_dist": 0.0},
    "clustering_dict": {"method": "dbscan", "eps": 0.095, "min_samples":19},
})


# nemi = NEMI(params={
#     "device": "gpu",
#     "embedding_dict":  {"n_components": 3, "n_neighbors": 57, "min_dist": 0.0},
#     "clustering_dict": {"method": "kmeans", "n_clusters": 300},
# })

# nemi = NEMI(params={
#     "device": "gpu",
#     "embedding_dict":  {"n_components": EMBEDDING_DIMENSIONS, "n_neighbors": 15, "min_dist": 0.0},
#     "clustering_dict": {"method": "agglomerative", "n_clusters": 20, "linkage": "single", "n_neighbors": 30},
# })


# single member to start; bump n (with assess_overlap=True) for the ensemble
nemi.run(patches, n=ENSEMBLE_MEMEBERS, output="nemi_out.npz", assess_overlap=True)

labels    = nemi.clusters      # (N,) cluster label per patch
embedding = nemi.embedding     # (N, 3) UMAP embedding
print("labels", labels.shape, "| embedding", embedding.shape)

  0%|          | 0/8 [00:00<?, ?it/s]

Fitting the embedding
[2026-08-03 13:42:30.152] [CUML] [warning] Spectral initialization failed, using random initialization instead.
Predicting the clusters
Clustering | device=gpu | {'method': 'dbscan', 'eps': 0.095, 'min_samples': 19}
[2026-08-03 13:42:37.879] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 13018 -> 10324. Using the larger integer type might result in better performance
Clusters found: 413
Sorting clusters


 12%|█▎        | 1/8 [04:10<29:14, 250.59s/it]

Fitting the embedding
[2026-08-03 13:42:41.882] [CUML] [info] Building knn graph using nn descent
[2026-08-03 13:42:46.403] [CUML] [warning] Spectral initialization failed, using random initialization instead.
Predicting the clusters
Clustering | device=gpu | {'method': 'dbscan', 'eps': 0.095, 'min_samples': 19}
[2026-08-03 13:42:47.128] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 13018 -> 10324. Using the larger integer type might result in better performance


 25%|██▌       | 2/8 [04:17<10:41, 106.99s/it]

...

 88%|████████▊ | 7/8 [04:49<00:15, 15.87s/it]

Clusters found: 447
Sorting clusters
Fitting the embedding
[2026-08-03 13:43:20.821] [CUML] [info] Building knn graph using nn descent
[2026-08-03 13:43:25.359] [CUML] [warning] Spectral initialization failed, using random initialization instead.
Predicting the clusters
Clustering | device=gpu | {'method': 'dbscan', 'eps': 0.095, 'min_samples': 19}
[2026-08-03 13:43:26.085] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 13018 -> 10324. Using the larger integer type might result in better performance


100%|██████████| 8/8 [04:56<00:00, 37.00s/it]

Clusters found: 465
Sorting clusters


465 465


In [ ]:
d = np.load("nemi_out.npz")
print(d.files) 

try :
    member_clusters = d["member_clusters"]  # (n, N)  per-member labels  <- entropy input
    embeddings      = d["embeddings"]       # (n, N, d) per-member UMAP embeddings
    clusters        = d["clusters"]         # (N,)   consensus labels (assess_overlap ran)
    embedding       = d["embedding"]        # (N, d) consensus/base embedding
    entropy         = d["entropy"]
    unassigned      = d["unassigned_frac"]
    
    print(embeddings.shape)
    print(member_clusters.shape)
    print(clusters.shape)
    print(embedding.shape)
    print(entropy.shape)
    print(unassigned.shape)

except :
    embeddings      = d["embedding"]       # (n, N, d) per-member UMAP embeddings
    clusters        = d["clusters"]  

In [ ]:
#labeled by consensus cluster
if ENSEMBLE_MEMEBERS > 1:
    vis.vis_dim_redux_list(embeddings, labels=clusters, dims=EMBEDDING_DIMENSIONS, titles=[f"member {i}" for i in range(len(embeddings))], alpha=0.4)
else:
    vis.vis_dim_redux(embeddings, labels=clusters, label_title="clusters", dims=EMBEDDING_DIMENSIONS, alpha=0.5)

In [ ]:
# vis.vis_dim_redux(embeddings[1], labels=entropy, dims=2, alpha=0.5, categorical=False, cmap="viridis")

vis.vis_dim_redux_list(d["embeddings"], labels=entropy, dims=EMBEDDING_DIMENSIONS, alpha=0.2,titles=[f"member {i}" for i in range(len(d["embeddings"]))], categorical=False, cmap="viridis")

In [ ]:
vis.vis_dim_redux_list(d["embeddings"], labels=unassigned, dims=EMBEDDING_DIMENSIONS, alpha = 0.5,
                   titles=[f"member {i}" for i in range(len(d["embeddings"]))], categorical=False, cmap="viridis")

In [ ]:
# colored by embedding's clustering
vis.vis_dim_redux_list(d["embeddings"], labels=d["member_clusters"], dims=EMBEDDING_DIMENSIONS, alpha=0.5,
                   titles=[f"member {i}" for i in range(len(d["embeddings"]))])

In [ ]:
np.unique(clusters)

In [ ]:
clusters=np.nan_to_num(clusters, nan=-1.0) # for a single ensemble member

vis.plot_global_cluster_maps(dataset, clusters, patch_size=8, alpha=0.1, point_size=10.0, extent=None,
                             coastlines=True, panel_size=8, drop_noise=False,
                             save_dir=None)

In [ ]:
vis.make_image_from_patches(dataset, clusters, patch_size=8, number_rows=20)

# Cluster composition

Where the ~450 DBSCAN clusters sit in the data, what they hold, and where they
are found.  `-1` (noise) is kept throughout as a cluster of its own.

In [ ]:
ids, pct = vis.plot_cluster_size_distribution(clusters)
print(f"noise (-1) holds {pct[ids == -1][0]:.1f}% | top 20 clusters cover {pct[:20].sum():.1f}%")

In [ ]:
# long tail: the same distribution on a log axis, largest 40 only
vis.plot_cluster_size_distribution(clusters, top_n=40, log=True)

In [ ]:
# feature spread within the largest clusters
vis.plot_cluster_feature_spread(dataset, clusters, features=["Theta", "Salt", "gradb2"],
                                patch_size=patch_size, top_n=15,
                                entropy=entropy, show_time=True)

In [ ]:
# same, for a random sample of clusters of any size
vis.plot_cluster_feature_spread(dataset, clusters, features=["Theta", "Salt", "gradb2"],
                                patch_size=patch_size, n_clusters=20, seed=0,
                                entropy=entropy, show_time=True)

In [ ]:
vis.plot_top_cluster_maps(dataset, clusters, patch_size=patch_size, top_n=12, n_cols=4)

In [ ]:
# prevalence instead of raw counts, which divides out the uneven sampling density
vis.plot_top_cluster_maps(dataset, clusters, patch_size=patch_size, top_n=12, n_cols=4,
                          normalize="fraction")

In [ ]:
# cutouts holding a given cluster, richest first; `cluster` is the actual label
# shown by the maps above, not a size rank
top_cluster = int(ids[ids >= 0][0])
vis.plot_cluster_cutouts(dataset, clusters, top_cluster,
                         features=["Theta", "Salt", "gradb2"],
                         patch_size=patch_size, number_rows=6)

In [ ]:
# any label works, and entropy adds its own column
vis.plot_cluster_cutouts(dataset, clusters, top_cluster,
                         features=["Theta", "gradb2"], patch_size=patch_size,
                         number_rows=4, start=6, entropy=entropy)

In [ ]:
# sampled log|∇b|² per timestamp, with cluster squares over it
vis.plot_global_field_cluster_maps(dataset, clusters, field="gradb2",
                                   patch_size=patch_size, panel_size=12)

In [ ]:
# same, overlaying only the 20 largest clusters so the colors stay readable
vis.plot_global_field_cluster_maps(dataset, clusters, field="gradb2",
                                   patch_size=patch_size, panel_size=12,
                                   clusters=ids[:20])